# Correcting tune 

##  Import of modules

In [ ]:
import jsons
import yaml

In [ ]:
from accml_lib.core.bl.delta_backend import StateCache
from accml_lib.core.bl.command_rewritter import CommandRewriter
from accml_lib.core.model.output.tune import Tune
from accml_lib.core.model.utils.command import ReadCommand

In [ ]:
from accml_lib.custom.bessyii.liasion_translator_setup import load_managers
from accml_lib.custom.bessyii.setup import setup

In [ ]:
from accml.custom.ophyd_async.ophyd_async_backend import OphydAsyncDeviceBackendRW
from accml.custom.ophyd_async.ophyd_async_delta_backend import OphydAsyncDeltaBackendRWProxy

In [ ]:
from accml.core.utils.simple_storage import SimpleDataStorage
from accml.core.utils.basic_measurement_execution_engine import BasicMeasurementExecutionEngine

In [ ]:
from accml.app.tune.model import TuneResponseCollection
from accml.app.tune.oracle import TuneOracle
from accml.app.tune.policy import TunePolicy
from accml.app.tune.tune_correction_controller import TuneCorrectionController

In [ ]:
import asyncio

## Setup of imputs

In [ ]:
!find  ../../ -name '*.yml';
!pwd

In [ ]:
with open("../../tune/tune_response_from_simulation.yml") as fp:
    d = yaml.load(fp, yaml.SafeLoader)
dm = jsons.load(d, TuneResponseCollection)

In [ ]:
yp, lm, ts = load_managers()
devices = setup(prefix=None)

In [ ]:
backend = OphydAsyncDeltaBackendRWProxy(
    OphydAsyncDeviceBackendRW(devices=devices),
    cache=StateCache(name="BESSY2_OphydAsync_Dev_State_Cache"),
)

In [ ]:
mexec = BasicMeasurementExecutionEngine(
    backend=backend,
    cmd_rewriter=CommandRewriter(liaison_manager=lm, translation_service=ts),
    storage=SimpleDataStorage(),
    expected_view_for_output="device",
    num_readings=2,
)

In [ ]:
tune_target = Tune(x=1065, y=855)

In [ ]:
oracle = TuneOracle(col=dm, target=tune_target)

In [ ]:
policy = TunePolicy(scale=1.0)

In [ ]:
controller = TuneCorrectionController(
    oracle=oracle,
    policy=policy,
    mexec=mexec,
    wait_after_set=1.2,
    wait_between_samples=0.5,
    n_samples=3
)

In [ ]:
rcmds = [ReadCommand(id="tune", property="transversal")]
set_cmds = [
        ReadCommand(id=elm.pc_name, property="delta_set_current") for elm in dm.col
]

In [ ]:
await asyncio.gather(*[devices.get(rc.id).connect() for rc in rcmds])

In [ ]:
await asyncio.gather(*[devices.get(rc.id).connect() for rc in set_cmds]);

In [ ]:
await controller.continuous(read_commands=rcmds, set_commands=set_cmds, n_steps=2)